# Task 2 Model 1: Random Forest

This notebook loads the prepared data exported by Notebook 1, engineers visual features,
trains Random Forest, and exports only this model's artefacts.


## How to Run

Run `01_task2_setup.ipynb` once before this notebook. This notebook does not repeat the split,
image decoding, normalisation, or baseline work.


## 1. Setup


In [ ]:
%matplotlib inline
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from src.task2_utils import (
    FEATURE_CONFIG, REPO_ROOT, TASK2_OUTPUT_DIR, ensure_task2_directories,
    export_model_results, extract_visual_features, per_class_table,
    prepared_namespace, result_frame,
)
ensure_task2_directories()


### 1.1 Random Forest configuration


In [ ]:
QUICK_RUN = True
RANDOM_STATE = 42
FEATURE_N_JOBS = -1
RF_N_ESTIMATORS = 100 if QUICK_RUN else 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 2
RF_MAX_FEATURES = "sqrt"
MODEL_PATH = REPO_ROOT / "models" / "task2_random_forest.joblib"


## 2. Load Prepared Data from Notebook 1


In [ ]:
data = prepared_namespace()
print(f"Loaded {len(data.train_frame):,} train and {len(data.validation_frame):,} validation rows")


## 3. Engineer Visual Features and Train Random Forest


In [ ]:
def feature_matrix(images, description):
    print(f"Extracting {description} features...")
    return np.vstack(Parallel(n_jobs=FEATURE_N_JOBS)(
        delayed(extract_visual_features)(image) for image in images
    ))

X_train_features = feature_matrix(data.x_train, "training")
X_val_features = feature_matrix(data.x_val, "validation")
print("Feature matrix:", X_train_features.shape, "using", FEATURE_CONFIG)


### 3.1 Random Forest training


In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF, max_features=RF_MAX_FEATURES,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
)
random_forest.fit(X_train_features, data.y_train)
joblib.dump(random_forest, MODEL_PATH)
rf_scores = random_forest.predict_proba(X_val_features)
rf_pred = random_forest.classes_[rf_scores.argmax(axis=1)]
display(result_frame(data.y_val, rf_pred, rf_scores, "Random Forest"))


### 3.2 Feature evidence


In [ ]:
importance = permutation_importance(
    random_forest, X_val_features, data.y_val, scoring="f1_macro",
    n_repeats=3 if QUICK_RUN else 8, random_state=RANDOM_STATE, n_jobs=-1,
)
feature_evidence = pd.DataFrame({
    "feature_index": np.arange(X_train_features.shape[1]),
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)
display(feature_evidence.head(20))
display(per_class_table(data.y_val, rf_pred, data.classes))


## 4. Export Random Forest Results


In [ ]:
output_dir = export_model_results(
    "random_forest", "Random Forest", data, rf_scores,
)
feature_evidence.to_csv(output_dir / "feature_importance.csv", index=False)
print("Saved model:", MODEL_PATH)
print("Saved outputs:", output_dir)
